# Movie Poster ResNet50 Embedding

- 본 노트북은 `movie_snapshot_enriched_utf8_sig.csv`의 `poster_url`을 이용해 영화 포스터 이미지를 다운로드하고, 사전학습 ResNet50으로 2048차원 이미지 embedding 벡터를 추출하는 baseline 실험입니다.
- 학습을 진행하지는 않으므로 grad를 계산하진 않습니다.

**확인해주세요**:
- ResNet50 기본 transform은 이미지를 224x224 중앙 crop으로 변환합니다.
    - 따라서 포스터의 상단/하단 일부가 잘립니다. 
- 포스터가 없는 영화의 처리 방식은 이후 회의에서 결정합시다.


## 1. 라이브러리 로드 및 드라이브 마운트
- 코랩 환경에서만 드라이브 마운트
- 로컬 환경일 시에도 ㄱㅊ
- 현재 런타임 cpu or gpu 여부 판단


In [27]:
from pathlib import Path
import hashlib
import time

import numpy as np
import pandas as pd
import requests
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models

try:
    from google.colab import drive
    drive.mount('/content/gdrive')
except ModuleNotFoundError:
    pass

print('torch:', torch.__version__)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')


torch: 2.12.0
device: cpu


## 2. CSV 로드
- csv 파일 경로도 하이브리드로 작동합니다.
    - 1순위: 구글드라이브
    - 2순위: 로컬의 프로젝트 루트 기준
    - 3순위: 해당 노트북 상대 경로 기준

In [28]:
csv_candidates = [
    Path('/content/gdrive/MyDrive/흥보위/sql_result/movie_snapshot_enriched_utf8_sig.csv'),
    Path('data/processed/movie_snapshot_enriched_utf8_sig.csv'),
    Path('../../data/processed/movie_snapshot_enriched_utf8_sig.csv'),
]

csv_file_path = next((path for path in csv_candidates if path.exists()), csv_candidates[0])
if csv_file_path is None:
    raise FileNotFoundError(
        'CSV 파일을 찾을 수 없습니다. csv_candidates 경로를 확인하세요.'
    )

df = pd.read_csv(csv_file_path, encoding='utf-8-sig')

print(f'CSV 경로: {csv_file_path}')
print(f'전체 행 수: {len(df):,}')
print(df.columns.tolist())
df.head()


CSV 경로: ../../data/processed/movie_snapshot_enriched_utf8_sig.csv
전체 행 수: 20,621
['movie_name_clean', 'release_date', 'cumulative_sales_amount', 'cumulative_audience', 'show_count', 'country', 'production_company', 'distributor', 'rating', 'genre', 'director', 'actor', 'kobis_movie_cd', 'kobis_match_status', 'kobis_match_rule', 'kobis_match_score', 'kmdb_movie_id', 'kmdb_movie_seq', 'kmdb_doc_id', 'kmdb_match_status', 'kmdb_match_rule', 'kmdb_match_score', 'poster_url', 'synopsis']


,movie_name_clean,release_date,cumulative_sales_amount,cumulative_audience,show_count,country,production_company,distributor,rating,genre,...,kobis_match_rule,kobis_match_score,kmdb_movie_id,kmdb_movie_seq,kmdb_doc_id,kmdb_match_status,kmdb_match_rule,kmdb_match_score,poster_url,synopsis
0,청춘의 십자로,1934-09-21,850000,170,2,한국,한국영상자료원,NaN,NaN,드라마,...,NaN,NaN,K,101.0,K00101,matched,exact_title_date,1.0,NaN,터널을 지나 서울역으로 진입한 기차에서 모녀 승객이 내리자 손님을 기다리던 한 청년...
1,미몽(죽음의 자장가),1936-10-25,52000,13,2,한국,경성촬영소,NaN,NaN,드라마,...,NaN,NaN,K,121.0,K00121,matched,exact_title_year_single_candidate,0.9,NaN,애순(문예봉)은 여염집의 부인으로 허영이 심하고 가정을 돌보지 않는다. 참다못한 남...
2,자유만세,1946-10-21,1268000,634,38,한국,고려영화(주),NaN,청소년관람불가,드라마,...,NaN,NaN,K,175.0,K00175,matched,exact_title_year_single_candidate,0.9,http://file.koreafilm.or.kr/thm/02/00/00/61/tn...,1945년 8월 서울. 독립운동을 하다 일제의 앞잡이 남부(독은기)의 배반으로 체포...
3,돈,1950-03-09,4282000,1844,66,한국,김프로덕션,NaN,NaN,"드라마,범죄",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,낙동강,1952-11-15,455000,91,1,한국,워너 브러더스 픽쳐스,NaN,NaN,드라마,...,NaN,NaN,K,246.0,K00246,matched,exact_title_date,1.0,NaN,대학을 졸업하고 낙동강 강변의 어느 자그마한 농촌의 고향으로 돌아온 일령은 그 고향...


## 2.5. 2016년 이후 데이터만 필터링


In [29]:
min_release_date = pd.Timestamp('2016-01-01')

df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
model_df = df[df['release_date'] >= min_release_date].copy()

print(f'전체 행 수: {len(df):,}')
print(f'2016년 이후 행 수: {len(model_df):,}')
print(f'제외된 행 수: {len(df) - len(model_df):,}')
model_df[['movie_name_clean', 'release_date', 'poster_url']].head()


전체 행 수: 20,621
2016년 이후 행 수: 18,003
제외된 행 수: 2,618


,movie_name_clean,release_date,poster_url
2618,섹스 잘하는 여자가 좋다,2016-01-02,NaN
2619,셜록: 유령신부,2016-01-02,http://file.koreafilm.or.kr/thm/02/00/04/18/tn...
2620,남편 제자와의 밀애,2016-01-03,NaN
2621,과외선생님 : 심화학습편,2016-01-05,NaN
2622,거유탐닉 - 그녀의 방에서,2016-01-06,NaN


## 3. poster_url 있는 행만 필터링


In [30]:
valid_poster_url = model_df['poster_url'].notna() & (model_df['poster_url'].astype(str).str.strip() != '')
poster_df = model_df[valid_poster_url].copy()
poster_df = poster_df.reset_index().rename(columns={'index': 'source_row_index'})

# 처음에는 100개 정도로 테스트하고, 전체 실행할 때 None으로 바꾼다.
MAX_IMAGES = 100
if MAX_IMAGES is not None:
    poster_df = poster_df.head(MAX_IMAGES).copy()

print(f'2016년 이후 행 수: {len(model_df):,}')
print(f'poster_url 있는 행 수: {valid_poster_url.sum():,}')
print(f'이번 실행 대상 행 수: {len(poster_df):,}')
poster_df[['source_row_index', 'movie_name_clean', 'release_date', 'kmdb_doc_id', 'poster_url']].head()


2016년 이후 행 수: 18,003
poster_url 있는 행 수: 3,597
이번 실행 대상 행 수: 100


,source_row_index,movie_name_clean,release_date,kmdb_doc_id,poster_url
0,2619,셜록: 유령신부,2016-01-02,F35167,http://file.koreafilm.or.kr/thm/02/00/04/18/tn...
1,2629,굿 다이노,2016-01-07,F35060,http://file.koreafilm.or.kr/thm/02/00/04/19/tn...
2,2638,짱구는 못말려 극장판: 나의 이사 이야기 선인장 대습격,2016-01-07,F35329,http://file.koreafilm.or.kr/thm/02/00/04/20/tn...
3,2639,포인트 브레이크,2016-01-07,F34728,http://file.koreafilm.or.kr/thm/02/00/04/19/tn...
4,2641,헤이트풀8,2016-01-07,F35295,http://file.koreafilm.or.kr/thm/02/00/04/19/tn...


## 4. 이미지 다운로드/cache


In [31]:
def safe_image_id(row):
    if pd.notna(row.get('kmdb_doc_id')) and str(row.get('kmdb_doc_id')).strip():
        return str(row['kmdb_doc_id']).strip().replace('/', '_')

    key = f"{row.get('movie_name_clean', '')}_{row.get('release_date', '')}_{row.get('source_row_index', '')}"
    return hashlib.md5(key.encode('utf-8')).hexdigest()


output_base_dir = csv_file_path.parent / 'poster_resnet50'
output_base_dir.mkdir(parents=True, exist_ok=True)

poster_dir = output_base_dir / 'posters'
poster_dir.mkdir(parents=True, exist_ok=True)

poster_df['poster_image_id'] = poster_df.apply(safe_image_id, axis=1)
poster_df['poster_download_url'] = poster_df['poster_url'].astype(str).str.strip()
poster_df['poster_path'] = poster_df['poster_image_id'].apply(lambda image_id: poster_dir / f'{image_id}.jpg')

print(f'결과 저장 폴더: {output_base_dir}')
print(f'포스터 저장 폴더: {poster_dir}')
poster_df[['movie_name_clean', 'poster_download_url', 'poster_path']].head()


결과 저장 폴더: ../../data/processed/poster_resnet50
포스터 저장 폴더: ../../data/processed/poster_resnet50/posters


,movie_name_clean,poster_download_url,poster_path
0,셜록: 유령신부,http://file.koreafilm.or.kr/thm/02/00/04/18/tn...,../../data/processed/poster_resnet50/posters/F...
1,굿 다이노,http://file.koreafilm.or.kr/thm/02/00/04/19/tn...,../../data/processed/poster_resnet50/posters/F...
2,짱구는 못말려 극장판: 나의 이사 이야기 선인장 대습격,http://file.koreafilm.or.kr/thm/02/00/04/20/tn...,../../data/processed/poster_resnet50/posters/F...
3,포인트 브레이크,http://file.koreafilm.or.kr/thm/02/00/04/19/tn...,../../data/processed/poster_resnet50/posters/F...
4,헤이트풀8,http://file.koreafilm.or.kr/thm/02/00/04/19/tn...,../../data/processed/poster_resnet50/posters/F...


In [32]:
def download_image(url, output_path, timeout=15, sleep_sec=0.1):
    output_path = Path(output_path)
    if output_path.exists() and output_path.stat().st_size > 0:
        return 'cached'

    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        response = requests.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()

        content_type = response.headers.get('Content-Type', '')
        if 'image' not in content_type.lower():
            return f'failed: non-image content-type={content_type}'

        output_path.write_bytes(response.content)
        time.sleep(sleep_sec)
        return 'downloaded'
    except Exception as exc:
        return f'failed: {type(exc).__name__}: {exc}'


download_results = []
for _, row in tqdm(poster_df.iterrows(), total=len(poster_df)):
    status = download_image(row['poster_download_url'], row['poster_path'])
    download_results.append(status)

poster_df['download_status'] = download_results
poster_df['download_success'] = poster_df['download_status'].isin(['cached', 'downloaded'])

poster_df['download_status'].value_counts().head(20)


100%|██████████| 100/100 [00:26<00:00,  3.78it/s]


download_status
downloaded    100
Name: count, dtype: int64

## 5. 다운로드 성공/실패 확인


In [33]:
success_df = poster_df[poster_df['download_success']].copy()
failed_df = poster_df[~poster_df['download_success']].copy()

print(f'다운로드 성공: {len(success_df):,}')
print(f'다운로드 실패: {len(failed_df):,}')

failed_df[['movie_name_clean', 'release_date', 'poster_download_url', 'download_status']].head(20)


다운로드 성공: 100
다운로드 실패: 0


,movie_name_clean,release_date,poster_download_url,download_status


## 6. 사전학습 CNN 모델 로드


In [34]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # cpu냐 gpu냐 그것이 문제로다

weights = models.ResNet50_Weights.DEFAULT # ResNet50 사전 기본 가중치 사용
cnn_model = models.resnet50(weights=weights)
cnn_model.fc = nn.Identity()  # 마지막 분류층을 제거하고 2048차원 embedding만 사용
cnn_model = cnn_model.to(device)
cnn_model.eval()

print('모델: ResNet50 pretrained')
print('embedding 차원: 2048')
print('device:', device)


모델: ResNet50 pretrained
embedding 차원: 2048
device: cpu


## 7. 이미지 전처리
- 이미지 크기 조정
- 중앙 crop
- tensor 변환
- 정규화

In [35]:
image_transform = weights.transforms() # 전처리 로드

class PosterDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = Image.open(row['poster_path']).convert('RGB') # RGB 형식 통일
        image_tensor = self.transform(image) # image -> tensor 변환
        return image_tensor, idx


poster_dataset = PosterDataset(success_df, image_transform)
poster_loader = DataLoader(
    poster_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

print(f'Embedding 추출 대상 이미지 수: {len(poster_dataset):,}')


Embedding 추출 대상 이미지 수: 100


## 8. CNN embedding 추출


In [36]:
all_embeddings = []
all_indices = []

with torch.no_grad():
    for images, indices in tqdm(poster_loader):
        images = images.to(device)
        embeddings = cnn_model(images).cpu().numpy()
        all_embeddings.append(embeddings)
        all_indices.extend(indices.numpy().tolist())

poster_embeddings = np.vstack(all_embeddings) if all_embeddings else np.empty((0, 2048), dtype=np.float32)
embedding_index_df = success_df.iloc[all_indices].reset_index(drop=True)

print('embedding shape:', poster_embeddings.shape)
embedding_index_df[['source_row_index', 'movie_name_clean', 'release_date', 'kmdb_doc_id', 'poster_path']].head()


100%|██████████| 4/4 [00:06<00:00,  1.70s/it]

embedding shape: (100, 2048)


,source_row_index,movie_name_clean,release_date,kmdb_doc_id,poster_path
0,2619,셜록: 유령신부,2016-01-02,F35167,../../data/processed/poster_resnet50/posters/F...
1,2629,굿 다이노,2016-01-07,F35060,../../data/processed/poster_resnet50/posters/F...
2,2638,짱구는 못말려 극장판: 나의 이사 이야기 선인장 대습격,2016-01-07,F35329,../../data/processed/poster_resnet50/posters/F...
3,2639,포인트 브레이크,2016-01-07,F34728,../../data/processed/poster_resnet50/posters/F...
4,2641,헤이트풀8,2016-01-07,F35295,../../data/processed/poster_resnet50/posters/F...


## 9. embedding 저장


In [37]:
embedding_npy_path = output_base_dir / 'poster_embeddings_resnet50.npy'
embedding_index_path = output_base_dir / 'poster_embeddings_resnet50_index.csv'
download_log_path = output_base_dir / 'poster_download_log.csv'

index_columns = [
    'source_row_index',
    'movie_name_clean',
    'release_date',
    'poster_path',
]
embedding_index_save_df = embedding_index_df[index_columns].copy()

np.save(embedding_npy_path, poster_embeddings)
embedding_index_save_df.to_csv(embedding_index_path, index=False, encoding='utf-8-sig')
poster_df.to_csv(download_log_path, index=False, encoding='utf-8-sig')

print(f'embedding 저장: {embedding_npy_path}')
print(f'index 저장: {embedding_index_path}')
print(f'download log 저장: {download_log_path}')
print(f'index 컬럼: {embedding_index_save_df.columns.tolist()}')


embedding 저장: ../../data/processed/poster_resnet50/poster_embeddings_resnet50.npy
index 저장: ../../data/processed/poster_resnet50/poster_embeddings_resnet50_index.csv
download log 저장: ../../data/processed/poster_resnet50/poster_download_log.csv
index 컬럼: ['source_row_index', 'movie_name_clean', 'release_date', 'poster_path']


## 10. 저장 결과 검증


In [38]:
loaded_embeddings = np.load(embedding_npy_path)
loaded_index_df = pd.read_csv(embedding_index_path, encoding='utf-8-sig')

print('loaded embedding shape:', loaded_embeddings.shape)
print('loaded index rows:', len(loaded_index_df))
print('shape/index match:', loaded_embeddings.shape[0] == len(loaded_index_df))
print('index columns:', loaded_index_df.columns.tolist())

loaded_index_df[['source_row_index', 'movie_name_clean', 'release_date', 'poster_path']].head()


loaded embedding shape: (100, 2048)
loaded index rows: 100
shape/index match: True
index columns: ['source_row_index', 'movie_name_clean', 'release_date', 'poster_path']


,source_row_index,movie_name_clean,release_date,poster_path
0,2619,셜록: 유령신부,2016-01-02,../../data/processed/poster_resnet50/posters/F...
1,2629,굿 다이노,2016-01-07,../../data/processed/poster_resnet50/posters/F...
2,2638,짱구는 못말려 극장판: 나의 이사 이야기 선인장 대습격,2016-01-07,../../data/processed/poster_resnet50/posters/F...
3,2639,포인트 브레이크,2016-01-07,../../data/processed/poster_resnet50/posters/F...
4,2641,헤이트풀8,2016-01-07,../../data/processed/poster_resnet50/posters/F...
